# Quantum Load Balancing for HPC Clusters — demo

Generates one scheduling instance, solves it exactly and with every classical baseline,
then shows where QAOA lands against them. Runs on a laptop, no GPU and no `cudaq`
needed — the QAOA part of the pipeline runs separately on GPU and is shown here from
`results/`/`figures/` already in the repo. See `README.md` for the full reproduction table.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

import problem as P
import baselines as B

## 1. One instance

12 jobs, lognormal runtimes, a planted-bipartition conflict graph (so a perfectly
balanced, zero-conflict split exists by construction).

In [ ]:
inst = P.generate_instance(n=12, seed=0)
print(f"n = {inst['n']} jobs, {len(inst['edges'])} conflict edges, lambda = {inst['lambda']}")
print("raw runtimes:", inst["weights_raw"])

## 2. Exact reference (brute force)

Every one of the 2^12 assignments, scored exactly — this is the ground truth every
`approx_ratio` in the project is measured against.

In [ ]:
ref = P.brute_force(inst)
print(f"C_min = {ref['c_min']:.4f}   C_max = {ref['c_max']:.4f}   "
      f"#optimal states = {len(ref['opt_states'])}")

## 3. Classical baselines

Karmarkar-Karp and greedy LPT only balance load (standard practice — they ignore
conflicts); simulated annealing and CP-SAT optimise the full cost, conflicts included.

In [ ]:
runners = {
    "Karmarkar-Karp": lambda: B.karmarkar_karp(inst),
    "Greedy LPT": lambda: B.greedy_lpt(inst),
    "Simulated annealing": lambda: B.simulated_annealing(inst, seed=0),
    "CP-SAT (exact)": lambda: B.cp_sat_solve(inst),
}

print(f"{'method':<22}{'cost':>10}{'violations':>12}{'approx_ratio':>15}")
for name, run in runners.items():
    r = run()
    ratio = P.approx_ratio(r["cost"], ref["c_min"], ref["c_max"])
    print(f"{name:<22}{r['cost']:>10.4f}{r['violations']:>12}{ratio:>15.4f}")

## 4. Where QAOA lands

QAOA itself runs only via CUDA-Q on a Linux machine (GPU or CPU target), so it isn't run
live in this notebook — this is the actual headline figure from the full pipeline
(`results/E1_depth.csv` + `results/baselines_timing.csv` through `make_figures.py`),
already checked into the repo.

In [ ]:
from IPython.display import Image
Image(filename=str(ROOT / "figures" / "qaoa_vs_baselines.png"))

QAOA (p = 6) beats both naive heuristics (KK, LPT) at almost every size tested, but not
the two baselines that actually optimise the full cost (SA, CP-SAT) — no quantum
advantage at these sizes, which is exactly what the project set out to check, not a
failure. See `report/` for the full discussion and `README.md` to regenerate every
figure, including this one, from scratch.